In [ ]:
# Lab 10, notebook 4: drift you cannot see in one day's numbers, and a record of every day to chart.
# Attached lakehouse: freshcart_lake. It calls the class embedding model as you (your sign-in, no key).
# Run it after lab10_3_health. Running it twice adds nothing twice.

### Settings

Paste the address your trainer gives you. The embedding model turns a text into a list of numbers that stands for its meaning. DRIFT_LIMIT is how far a day's meaning may move from the 30 days before it; we set 0.02 from our normal days.

In [ ]:
import hashlib, json, math, time, urllib.request
import mlflow
from pyspark.sql import functions as F

FOUNDRY = "PASTE-THE-ADDRESS-YOUR-TRAINER-GIVES-YOU"   # your trainer gives you this address
EMBEDDING_MODEL = "text-embedding-3-small"
EXPERIMENT = "lab10_pipeline_health"     # MLflow makes this experiment item in your workspace
DRIFT_LIMIT = 0.02                       # cosine distance from the last 30 days; set from our normal days, see below
FIRST_DAY = "2024-12-31"                 # the first day added in class; earlier days are the history

def table(name):
    """A table if it exists yet, else None."""
    return spark.read.table(name) if spark.catalog.tableExists(name) else None

### 1. Embedding drift: has the meaning of the tickets moved? (no labels needed)

Each distinct ticket text is turned into numbers once and kept in text_embeddings, so no text is sent twice. For each day it takes the average meaning of that day's tickets and of the 30 days before, and measures the distance between them: 0 means the same, and above 0.02 is an alert. The result goes to drift_checks.

In [ ]:
# Each distinct text is embedded once and kept, keyed on model and text, like the labels in Lab 9.
def embed(texts):
    token = notebookutils.credentials.getToken("https://cognitiveservices.azure.com")
    body = json.dumps({"input": texts}).encode()
    request = urllib.request.Request(f"{FOUNDRY}/openai/deployments/{EMBEDDING_MODEL}/embeddings?api-version=2024-10-21",
                                     body, {"Authorization": "Bearer " + token, "Content-Type": "application/json"})
    return [d["embedding"] for d in json.load(urllib.request.urlopen(request, timeout=60))["data"]]

clean = spark.read.table("tickets_clean").withColumn("d", F.to_date("created_at"))
texts = sorted(r["text"] for r in clean.select("text").distinct().collect())
kept = table("text_embeddings")
known = {r["key"] for r in kept.collect()} if kept is not None else set()
key = lambda t: hashlib.sha256(f"{EMBEDDING_MODEL}|{t}".encode()).hexdigest()
new = [t for t in texts if key(t) not in known]
if new:
    rows = [{"key": key(t), "model": EMBEDDING_MODEL, "text": t, "vector": v} for t, v in zip(new, embed(new))]
    spark.createDataFrame(rows).write.format("delta").mode("append").saveAsTable("text_embeddings")
print(f"{len(texts)} distinct texts, {len(new)} embedded now, {len(texts) - len(new)} from the store")
vectors = {r["text"]: r["vector"] for r in spark.read.table("text_embeddings").filter(F.col("model") == EMBEDDING_MODEL).collect()}

def centre(counts):
    """The average meaning of a set of tickets: each text's vector, weighted by how many tickets have it."""
    total = sum(counts.values())
    return [sum(vectors[t][i] * n for t, n in counts.items()) / total for i in range(len(next(iter(vectors.values()))))]

def distance(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return 1 - dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

per_day = {}
for r in clean.groupBy("d", "text").count().collect():
    per_day.setdefault(r["d"], {})[r["text"]] = r["count"]
days = sorted(d for d in per_day if str(d) >= FIRST_DAY)
drift = {}
for day in days:
    history = {}
    for d, counts in per_day.items():
        if 0 < (day - d).days <= 30:
            for t, n in counts.items():
                history[t] = history.get(t, 0) + n
    drift[day] = round(distance(centre(per_day[day]), centre(history)), 4)
report = spark.createDataFrame([(str(d), v, "ALERT" if v > DRIFT_LIMIT else "OK") for d, v in drift.items()],
                               "day string, distance double, status string")
report.write.format("delta").mode("overwrite").saveAsTable("drift_checks")   # worked out again in full each run
display(report)

### 2. One MLflow run per day, so the experiment charts every day side by side

MLflow keeps a record of runs and their numbers. For each day not yet recorded, this cell writes one run with five numbers: tickets, quarantine rate, late-delivery share, embedding drift and health alerts. Days already recorded are skipped, so running it twice adds nothing twice.

In [ ]:
mlflow.set_experiment(EXPERIMENT)
done = {r.data.tags["day"] for r in mlflow.search_runs(experiment_names=[EXPERIMENT], output_format="list")
        if r.data.tags.get("day")}
raw = spark.read.table("tickets_raw").withColumn("d", F.to_date("created_at"))
parked = spark.read.table("tickets_quarantine").withColumn("d", F.to_date("created_at"))
enriched = spark.read.table("tickets_enriched").withColumn("d", F.to_date("created_at"))
health = table("health_checks")
if health is not None:                                 # the health notebook may run a day more than once: keep its last run
    latest = health.groupBy("day").agg(F.max("run_at").alias("run_at"))
    health = health.join(latest, ["day", "run_at"])
logged = 0
for day in days:
    if str(day) in done:                               # already recorded: running twice adds nothing twice
        continue
    tickets = raw.filter(F.col("d") == day).count()
    labelled = enriched.filter((F.col("d") == day) & F.col("ai_category").isNotNull())
    late = labelled.filter(F.col("ai_category") == "late_delivery").count() / max(labelled.count(), 1)
    alerts = (health.filter((F.col("day") == str(day)) & (F.col("status") == "ALERT")).select("check").distinct().count()
              if health is not None else 0)
    with mlflow.start_run(run_name=f"day {day}"):
        mlflow.set_tags({"day": str(day), "embedding_model": EMBEDDING_MODEL})
        mlflow.log_metrics({"tickets": tickets, "quarantine_rate": parked.filter(F.col("d") == day).count() / max(tickets, 1),
                            "late_delivery_share": late, "embedding_drift": drift[day], "health_alerts": alerts})
    logged += 1
print(f"{logged} day(s) recorded now, {len(done)} were recorded before. Open the experiment {EXPERIMENT} in your workspace.")